In [1]:
# =========================================================================
# COMPLETE PHASE 3 KAGGLE SCRIPT (PASTE THIS ENTIRE FILE INTO A NEW CELL)
# =========================================================================
!pip install thop

# ==========================================
# CELL 1: ENVIRONMENT SETUP & IMPORTS
# ==========================================
# Run this cell first to install required packages and import dependencies.

# Install specialized packages not pre-loaded on Kaggle
# Note: mamba-ssm is excluded due to Kaggle C++ compilation issues; the model will fallback to LSTM.
!pip install -q tifffile spectral

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import random
import joblib
import scipy

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score
import xgboost as xgb
import lightgbm as lgb

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from einops import rearrange

# Explainable AI
import shap

# Kaggle Output Directory
OUTPUT_DIR = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set random seed for reproducibility (will be dynamically changed in 10-seed loop)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

import sys
import psutil
import torch
import platform
import gc

def print_system_specs():
    print("="*50)
    print("💻 SYSTEM SPECIFICATIONS & ENVIRONMENT LOG")
    print("="*50)
    print(f"Python Version: {sys.version.split(' ')[0]}")
    print(f"OS: {platform.system()} {platform.release()}")
    
    # RAM
    ram = psutil.virtual_memory()
    print(f"System RAM: {ram.total / (1024**3):.2f} GB (Available: {ram.available / (1024**3):.2f} GB)")
    
    # GPU
    if torch.cuda.is_available():
        print(f"PyTorch Version: {torch.__version__} (CUDA: {torch.version.cuda})")
        print(f"GPUs Available: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            prop = torch.cuda.get_device_properties(i)
            print(f"  [{i}] {prop.name} - VRAM: {prop.total_memory / (1024**3):.2f} GB")
    else:
        print("❌ CRITICAL: NO GPU DETECTED! Ensure you selected GPU T4x2 in Kaggle settings.")
    print("="*50 + "\n")

print_system_specs()

print("✅ Cell 1 Complete: Environment configured and libraries imported.")


# ==========================================
# CELL 2: DATASET EXTRACTION & PREPROCESSING
# ==========================================

# 1. Reverse-Engineer Local Dataset (.bil files)
def read_bil_file(filepath, samples=320, bands=168, dtype=np.uint16):
    filesize = os.path.getsize(filepath)
    bytes_per_pixel = np.dtype(dtype).itemsize
    lines = filesize / (samples * bands * bytes_per_pixel)
    if not lines.is_integer():
        return None
    lines = int(lines)
    raw_data = np.fromfile(filepath, dtype=dtype)
    img_cube = raw_data.reshape((lines, bands, samples))
    img_cube = np.transpose(img_cube, (0, 2, 1)) # (H, W, C)
    return img_cube

def extract_patches(img_cube, label, condition, patch_size=15):
    h, w, c = img_cube.shape
    patches, labels, conditions = [], [], []
    for i in range(0, h - patch_size, patch_size):
        for j in range(0, w - patch_size, patch_size):
            patch = img_cube[i:i+patch_size, j:j+patch_size, :]
            patches.append(patch)
            labels.append(label)
            conditions.append(condition) # 0=Dry, 1=Moist
    return patches, labels, conditions

def load_local_dataset(base_dir):
    print(f"⏳ Extracting Local Dataset from: {base_dir}")
    X, y, cond = [], [], []
    
    # Define mapping based on user directory structure
    class_map = {'Black Soil': 0, 'Red Soil': 1, 'Yellow Soil': 2}
    
    bil_files = []
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.endswith('.bil'):
                bil_files.append(os.path.join(root, f))
                
    if len(bil_files) == 0:
        raise FileNotFoundError("❌ CRITICAL: 0 .bil files found! You MUST click 'Add Data' -> 'Your Datasets' -> 'local-soil-hyperspectral-dataset' in your Kaggle Notebook to mount the local dataset!")
        
    print(f"Found {len(bil_files)} .bil files. Extracting patches...")
    
    for filepath in bil_files:
        root = os.path.dirname(filepath)
        img = read_bil_file(filepath)
        if img is None: continue
        
        # Assign labels
        label = -1
        if 'black' in root.lower(): label = 0
        elif 'red' in root.lower(): label = 1
        elif 'yellow' in root.lower(): label = 2
        
        condition = 1 if 'moist' in root.lower() else 0
        
        if label != -1:
            p, l, c = extract_patches(img, label, condition)
            X.extend(p)
            y.extend(l)
            cond.extend(c)
                    
    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    cond = np.array(cond)
    print(f"✅ Local Data Loaded: {X.shape[0]} patches extracted. Shape: {X.shape}")
    return X, y, cond

import scipy.io as sio

import urllib.request

def load_public_datasets(kaggle_input_dir="/kaggle/input"):
    print("⏳ Downloading and Loading Indian Pines Dataset...")
    public_data = {}
    
    # Using stable GitHub mirror because the official EHU server is frequently down (HTTP 503)
    data_url = "https://raw.githubusercontent.com/gokriznastic/HybridSN/master/data/Indian_pines_corrected.mat"
    label_url = "https://raw.githubusercontent.com/gokriznastic/HybridSN/master/data/Indian_pines_gt.mat"
    
    data_path = "Indian_pines_corrected.mat"
    label_path = "Indian_pines_gt.mat"
    
    try:
        if not os.path.exists(data_path):
            print("Downloading Indian Pines Data...")
            urllib.request.urlretrieve(data_url, data_path)
        if not os.path.exists(label_path):
            print("Downloading Indian Pines Labels...")
            urllib.request.urlretrieve(label_url, label_path)
            
        import scipy.io as sio
        X_full = sio.loadmat(data_path)['indian_pines_corrected'].astype(np.float32)
        y_full = sio.loadmat(label_path)['indian_pines_gt']
        
        # Flatten into valid pixels for standard training
        X_pixels = X_full.reshape(-1, X_full.shape[2])
        y_pixels = y_full.reshape(-1)
        
        valid_idx = np.where(y_pixels > 0)[0]
        X_valid = X_pixels[valid_idx]
        y_valid = y_pixels[valid_idx] - 1  # 0-indexed labels
        
        public_data['IndianPines'] = {
            'X': X_valid, 
            'y': y_valid, 
            'bands': X_valid.shape[1], 
            'classes': len(np.unique(y_valid)),
            'img': X_full # Save original 3D cube for spatial mapping
        }
        print(f"✅ Indian Pines Loaded: {X_valid.shape[0]} valid pixels, {X_valid.shape[1]} bands.")
    except Exception as e:
        print(f"⚠️ Indian Pines failed to load: {e}")
        
    return public_data

class SoilDataset(Dataset):
    def __init__(self, X, y, indices=None):
        self.X = X
        self.y = y
        self.indices = indices if indices is not None else np.arange(len(X))
        
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x = torch.as_tensor(self.X[real_idx], dtype=torch.float32)
        # Normalize spectra (Standard Normal Variate) on the fly
        mean = x.mean()
        std = x.std() + 1e-8
        x = (x - mean) / std
        y_val = torch.as_tensor(self.y[real_idx], dtype=torch.long)
        return x, y_val

print("✅ Cell 2 Complete: Data pipeline ready.")


# ==========================================
# CELL 3: MODEL ARCHITECTURES (6 FAMILIES)
# ==========================================

# 1. 1D-CNN (Basic Deep Learning)
class CNN1D(nn.Module):
    def __init__(self, bands=168, classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(bands, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.fc = nn.Linear(128, classes)
        
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :] # Shape (B, C)
        x = x.unsqueeze(2) # (B, C, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.squeeze(2)
        return self.fc(x)

# 2. 3D-CNN (Spatial-Spectral)
class CNN3D(nn.Module):
    def __init__(self, bands=168, classes=3):
        super().__init__()
        # Input: (B, C, Depth=bands, H, W)
        self.conv1 = nn.Conv3d(1, 8, kernel_size=(7, 3, 3), padding=(0,1,1))
        self.conv2 = nn.Conv3d(8, 16, kernel_size=(5, 3, 3), padding=(0,1,1))
        self.fc = nn.Linear(16 * (bands - 10) * 15 * 15, classes)
        
    def forward(self, x):
        # x: (B, H, W, C) -> (B, 1, C, H, W)
        x = x.permute(0, 3, 1, 2).unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.flatten(1)
        return self.fc(x)

# 3. 1D-ResNet
class ResNet1DBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
    def forward(self, x):
        return F.relu(self.conv2(F.relu(self.conv1(x))) + x)

class ResNet1D(nn.Module):
    def __init__(self, bands=168, classes=3):
        super().__init__()
        self.conv_in = nn.Conv1d(bands, 64, kernel_size=3, padding=1)
        self.res1 = ResNet1DBlock(64)
        self.res2 = ResNet1DBlock(64)
        self.fc = nn.Linear(64, classes)
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = x.unsqueeze(2)
        x = F.relu(self.conv_in(x))
        x = self.res2(self.res1(x)).squeeze(2)
        return self.fc(x)

# 4. SOTA: Vision Transformer (ViT) / Self-Attention
class ViT1D(nn.Module):
    def __init__(self, bands=168, classes=3, dim=128, depth=4, heads=4):
        super().__init__()
        self.patch_embed = nn.Linear(1, dim)
        self.pos_embed = nn.Parameter(torch.randn(1, bands, dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.fc = nn.Linear(dim, classes)
        
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = x.unsqueeze(-1) # (B, Bands, 1)
        x = self.patch_embed(x) + self.pos_embed
        x = self.transformer(x)
        x = x.mean(dim=1) # Global Average Pooling
        return self.fc(x)

# 5. Graph Convolutional Network (GCN) Stub
class GCN1D(nn.Module):
    def __init__(self, bands=168, classes=3):
        super().__init__()
        # A simple spectral GCN approximation using Conv1D over spectral graphs
        self.conv1 = nn.Conv1d(bands, 64, kernel_size=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=1)
        self.fc = nn.Linear(128, classes)
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = x.unsqueeze(2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return self.fc(x.squeeze(2))

# 6. Mamba State-Space Model
class Mamba1D(nn.Module):
    def __init__(self, bands=168, classes=3, dim=128):
        super().__init__()
        self.proj = nn.Linear(1, dim)
        try:
            from mamba_ssm import Mamba
            self.mamba = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
            self.is_lstm = False
        except ImportError:
            # Fallback to simple RNN/LSTM if mamba fails to import on Kaggle
            self.mamba = nn.LSTM(dim, dim, batch_first=True)
            self.is_lstm = True
        self.fc = nn.Linear(dim, classes)
        
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = x.unsqueeze(-1)
        x = self.proj(x)
        if self.is_lstm:
            x, _ = self.mamba(x)
        else:
            x = self.mamba(x)
        x = x.mean(dim=1)
        return self.fc(x)

# 7. Ablation Model: ViT Without Attention (Just an MLP on the patches)
class ViT_No_Attention(nn.Module):
    def __init__(self, bands=168, classes=3, dim=128):
        super().__init__()
        self.fc1 = nn.Linear(bands, dim)
        self.fc2 = nn.Linear(dim, classes)
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = F.relu(self.fc1(x))
        return self.fc2(x)

print("✅ Cell 3 Complete: Architectures defined.")


def train_pytorch_model(model, train_loader, test_loader, epochs=100, patience=10, class_weights=None):
    import copy
    import time
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    if class_weights is not None:
        weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
        criterion = FocalLoss(alpha=weights_tensor)
    else:
        # Standard loss is perfect for perfectly balanced 50/50 Sampler batches
        criterion = nn.CrossEntropyLoss()
        
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    best_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        correct_train = 0
        total_train = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
            preds = torch.argmax(out, dim=1)
            correct_train += (preds == y_batch).sum().item()
            total_train += y_batch.size(0)
            
        train_loss = epoch_loss / len(train_loader)
        train_acc = correct_train / total_train
        scheduler.step()
        
        # Validation for early stopping
        model.eval()
        val_loss = 0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                out = model(X_batch)
                val_loss += criterion(out, y_batch).item()
                preds = torch.argmax(out, dim=1)
                correct_val += (preds == y_batch).sum().item()
                total_val += y_batch.size(0)
                
        val_loss /= len(test_loader)
        val_acc = correct_val / total_val
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"      Early stopping triggered at epoch {epoch+1}")
            break
            
    # Restore best weights
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
            
    # Pure Inference Evaluation Loop
    model.eval()
    all_preds, all_y = [], []
    inf_start = time.time()
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            out = model(X_batch.to(device))
            preds = torch.argmax(out, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_y.extend(y_batch.numpy())
    inf_end = time.time()
    inference_time = inf_end - inf_start
            
    acc = accuracy_score(all_y, all_preds)
    kappa = cohen_kappa_score(all_y, all_preds)
    return acc, kappa, np.array(all_preds), model, history, inference_time

def execute_10_seed_training(X, y=None, conditions=None, dataset_name="Dataset", bands=168, classes=3, num_seeds=3, train_size=0.8, test_size=0.2, epochs=100, patience=20):
    print(f"\n⏳ Starting {num_seeds}-Seed Rigorous Training Pipeline for {dataset_name}...")
    is_lazy = isinstance(X, Dataset)
    
    # Exclude Sklearn models for lazy datasets, and 3D models for 1D datasets
    if is_lazy:
        models_to_test = ['1D-CNN', 'GCN', 'Mamba', 'ViT']
    elif X.ndim < 4:
        models_to_test = ['SVM', 'LGBM', 'XGBoost', '1D-CNN', 'GCN', 'Mamba', 'ViT']
    else:
        models_to_test = ['SVM', 'LGBM', 'XGBoost', '1D-CNN', '3D-CNN', 'GCN', 'Mamba', 'ViT']
        
    results = {m: {'acc': [], 'kappa': []} for m in models_to_test}
    
    from sklearn.utils.class_weight import compute_class_weight
    
    for seed in range(num_seeds):
        seed_everything(seed)
        print(f"\n--- Running Seed {seed+1}/{num_seeds} ---")
        
        if is_lazy:
            # Extract lazy labels for stratify & weighting
            if hasattr(X, 'df'):
                y_all = (X.df['SOIL'] > 0).astype(int).values
            elif hasattr(X, 'labels') and X.labels is not None:
                y_all = X.labels[:, :, 0].flatten()
            else:
                y_all = None

            indices = list(range(len(X)))
            
            # Auto-Scale Split Size: Prevent starvation on tiny datasets, prevent timeouts on massive datasets
            dynamic_train_size = 0.8 if len(X) < 5000 else 0.05
            print(f"📊 Auto-Scaling Split: Using {dynamic_train_size*100:.1f}% for training ({int(len(X) * dynamic_train_size)} samples)")
            
            # Stratified Split
            if y_all is not None:
                # Grab EXACTLY dynamic_train_size and perfectly stratify
                train_idx, temp_idx, y_train_weights, temp_y = train_test_split(indices, y_all, train_size=dynamic_train_size, stratify=y_all, random_state=seed)
                # Grab EXACTLY test_size of the total length from the remainder
                test_ratio = min(1.0, test_size / (1.0 - dynamic_train_size))
                if test_ratio == 1.0:
                    test_idx = temp_idx
                else:
                    test_idx, _, _, _ = train_test_split(temp_idx, temp_y, train_size=test_ratio, stratify=temp_y, random_state=seed)
            else:
                np.random.shuffle(indices)
                tr_split = int(train_size * len(X))
                te_split = tr_split + int(test_size * len(X))
                train_idx = indices[:tr_split]
                test_idx = indices[tr_split:te_split]
                y_train_weights = None
            
            # Bulletproof Sampler: guarantees 50/50 batches regardless of data volume
            if y_train_weights is not None:
                unique_classes, counts = np.unique(y_train_weights, return_counts=True)
                # Ensure counts avoids divide by zero
                weight_dict = {cls: (1.0 / max(c, 1)) for cls, c in zip(unique_classes, counts)}
                sample_weights = [weight_dict[y] for y in y_train_weights]
                sampler = torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
                train_loader = DataLoader(torch.utils.data.Subset(X, train_idx), batch_size=16, sampler=sampler)
            else:
                train_loader = DataLoader(torch.utils.data.Subset(X, train_idx), batch_size=16, shuffle=True)
                
            test_loader = DataLoader(torch.utils.data.Subset(X, test_idx), batch_size=16, shuffle=False)
            X_train, y_train, X_test, y_test = None, None, None, None
        else:
            if conditions is None:
                # Standard random stratified split if no conditions exist (e.g. Indian Pines)
                dynamic_train_size = 0.8 if len(X) < 5000 else 0.05
                print(f"📊 Auto-Scaling Split: Using {dynamic_train_size*100:.1f}% for training ({int(len(X) * dynamic_train_size)} samples)")
                indices = np.arange(len(X))
                try:
                    train_idx, test_idx = train_test_split(indices, train_size=dynamic_train_size, stratify=y, random_state=seed)
                except ValueError:
                    # Fallback for Indian Pines if minority classes (like Oats) are too small to stratify at 5%
                    print("⚠️ Stratified split failed (likely due to tiny minority classes). Falling back to random split.")
                    train_idx, test_idx = train_test_split(indices, train_size=dynamic_train_size, random_state=seed)
            else:
                # Split: Train on Dry (condition 0) + 10% Moist. Test exclusively on remaining Moist.
                moist_indices = np.where(conditions == 1)[0]
                dry_indices = np.where(conditions == 0)[0]
                
                train_moist, test_moist = train_test_split(moist_indices, test_size=test_size, random_state=seed)
                train_idx = np.concatenate([dry_indices, train_moist])
                test_idx = test_moist
            
            # PyTorch Loaders (Zero-Copy)
            train_loader = DataLoader(SoilDataset(X, y, train_idx), batch_size=32, shuffle=True)
            test_loader = DataLoader(SoilDataset(X, y, test_idx), batch_size=32, shuffle=False)
            y_train_weights = y[train_idx]
            
        # Calculate Class Weights
        if y_train_weights is not None:
            unique_classes = np.unique(y_train_weights)
            c_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train_weights)
            # Ensure weight array covers all expected classes
            full_weights = np.ones(classes)
            for i, cls in enumerate(unique_classes):
                if cls < classes:
                    full_weights[cls] = c_weights[i]
            c_weights = full_weights
        else:
            c_weights = None
        
        for m in models_to_test:
            if m in ['SVM', 'LGBM', 'XGBoost', 'MLP'] and not is_lazy:
                acc, kap, preds, _ = train_sklearn_model(m, X, y, train_idx, test_idx)
            else:
                if m == '1D-CNN': net = CNN1D(bands=bands, classes=classes)
                elif m == '3D-CNN': net = CNN3D(bands=bands, classes=classes)
                elif m == 'GCN': net = GCN1D(bands=bands, classes=classes)
                elif m == 'Mamba': net = Mamba1D(bands=bands, classes=classes)
                elif m == 'ViT': net = ViT1D(bands=bands, classes=classes)
                
                # Prevent Double-Dipping: If using WeightedRandomSampler (is_lazy), data is already 50/50.
                pytorch_weights = None if is_lazy else c_weights
                
                acc, kap, preds, _, _, _ = train_pytorch_model(net, train_loader, test_loader, epochs=epochs, patience=patience, class_weights=pytorch_weights)
                
            results[m]['acc'].append(acc)
            results[m]['kappa'].append(kap)
            print(f"{m:10s} -> Acc: {acc:.4f} | Kappa: {kap:.4f}")
            
            
    # Aggregate Results
    print("\n" + "="*50)
    print(f"FINAL {num_seeds}-SEED RESULTS ({dataset_name})")
    print("="*50)
    final_df = []
    for m in models_to_test:
        mean_acc = np.mean(results[m]['acc']) * 100
        std_acc = np.std(results[m]['acc']) * 100
        mean_kap = np.mean(results[m]['kappa'])
        std_kap = np.std(results[m]['kappa'])
        final_df.append({'Model': m, 'OA (%)': f"{mean_acc:.2f} ± {std_acc:.2f}", 'Kappa': f"{mean_kap:.4f} ± {std_kap:.4f}"})
        
    df = pd.DataFrame(final_df)
    print(df.to_string(index=False))
    
    out_csv = os.path.join(OUTPUT_DIR, f"Table2_{num_seeds}Seed_Results_{dataset_name.replace(' ', '_')}.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n✅ Results saved to {out_csv}")
    return results

def execute_ablation_studies(X, y, conditions):
    print("\n⏳ Running Ablation Studies (Removing Attention Block)...")
    moist_indices = np.where(conditions == 1)[0]
    dry_indices = np.where(conditions == 0)[0]
    
    train_moist, test_moist = train_test_split(moist_indices, test_size=0.8, random_state=42)
    train_idx = np.concatenate([dry_indices, train_moist])
    test_idx = test_moist
    
    train_loader = DataLoader(SoilDataset(X, y, train_idx), batch_size=32, shuffle=True)
    test_loader = DataLoader(SoilDataset(X, y, test_idx), batch_size=32, shuffle=False)
    
    print("Training ViT WITH Attention...")
    acc_att, _, _, _, _, _ = train_pytorch_model(ViT1D(), train_loader, test_loader, epochs=15)
    
    print("Training ViT WITHOUT Attention...")
    acc_no_att, _, _, _, _, _ = train_pytorch_model(ViT_No_Attention(), train_loader, test_loader, epochs=15)
    
    
    print("\n" + "="*50)
    print("TABLE 3: ABLATION RESULTS")
    print(f"ViT (With Attention):    {acc_att*100:.2f}% Accuracy")
    print(f"ViT (Without Attention): {acc_no_att*100:.2f}% Accuracy")
    print(f"Drop without Attention:  {(acc_att - acc_no_att)*100:.2f}%")
    print("="*50)

print("✅ Cell 4 Complete: 10-seed training loop defined.")


# ==========================================
# EXP 2 - CELL 1: HybridSN Architecture & FLOPs
# ==========================================
# Run this command in a Kaggle cell first:
# !pip install thop

import torch
import torch.nn as nn
import torch.nn.functional as F
from thop import profile
from thop import clever_format

class HybridSN(nn.Module):
    """
    HybridSN: Exploring 3D-2D CNN Feature Hierarchy for Hyperspectral Image Classification
    Inputs: [B, 1, Bands, H, W] (Usually H=15, W=15 or 25x25)
    """
    def __init__(self, bands=200, classes=16, spatial_size=15):
        super(HybridSN, self).__init__()
        # 3D Convolutional Blocks
        self.conv1 = nn.Conv3d(in_channels=1, out_channels=8, kernel_size=(7, 3, 3))
        self.conv2 = nn.Conv3d(in_channels=8, out_channels=16, kernel_size=(5, 3, 3))
        self.conv3 = nn.Conv3d(in_channels=16, out_channels=32, kernel_size=(3, 3, 3))
        
        # Calculate shape after 3D Convs
        # (bands-7+1 -5+1 -3+1) = bands-12
        # spatial: 15-3+1 = 13; 13-3+1 = 11; 11-3+1 = 9
        out_bands = bands - 12
        out_spatial = spatial_size - 6
        
        # 2D Convolutional Block
        # Reshape: [B, 32, out_bands, out_spatial, out_spatial] -> [B, 32 * out_bands, out_spatial, out_spatial]
        self.conv4 = nn.Conv2d(in_channels=32 * out_bands, out_channels=64, kernel_size=(3, 3))
        
        # Calculate shape after 2D Conv
        out_spatial2d = out_spatial - 2
        self.flat_features = 64 * out_spatial2d * out_spatial2d
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(self.flat_features, 256)
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.4)
        self.fc3 = nn.Linear(128, classes)

    def forward(self, x):
        # Input shape: (B, H, W, C) from our standard loader
        # Convert to (B, 1, C, H, W)
        x = x.permute(0, 3, 1, 2).unsqueeze(1)
        
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        
        # Reshape for 2D Conv
        b, c, d, h, w = x.shape
        x = x.view(b, c * d, h, w)
        
        x = F.relu(self.conv4(x))
        x = x.flatten(1)
        
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

# ==========================================
# FLOPs Calculation Script
# ==========================================
def calculate_flops_and_params(model, input_shape):
    dummy_input = torch.randn(input_shape).to(next(model.parameters()).device)
    flops, params = profile(model, inputs=(dummy_input,), verbose=False)
    flops_str, params_str = clever_format([flops, params], "%.2f")
    print(f"{model.__class__.__name__} -> FLOPs: {flops_str} | Params: {params_str}")

# Example Usage:
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. ViT Input: (Batch, H, W, Bands) - Indian Pines
    vit_input = (1, 15, 15, 200)
    vit = ViT1D(bands=200, classes=16).to(device) # Assuming ViT1D is defined in cell_3
    
    # 2. HybridSN Input: (Batch, H, W, Bands) - Indian Pines
    hsn_input = (1, 15, 15, 200)
    hsn = HybridSN(bands=200, classes=16, spatial_size=15).to(device)
    
    print("--- FLOPs Benchmark ---")
    calculate_flops_and_params(vit, vit_input)
    calculate_flops_and_params(hsn, hsn_input)
    
    print("\nNote: LightGBM FLOPs are negligible (Tree Splits based on depth), simply report as 'N/A (Tree Splits)'")


# ==========================================
# EXP 2 - CELL 2: Few-Shot Curves & F1 Metrics (Indian Pines Only)
# ==========================================
# Note: Ensure you have your `X` (patches), `y` (labels) and Indian Pines `bands` and `classes` defined before running this.
# Also ensure you import `accuracy_score`, `f1_score`, `confusion_matrix`, `classification_report` from `sklearn.metrics`.

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def calc_average_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    class_accuracies = cm.diagonal() / cm.sum(axis=1)
    # Ignore NaN if a class is entirely missing from test set
    return np.nanmean(class_accuracies) * 100


def plot_training_curves(history, model_name, split):
    plt.figure(figsize=(12, 5))
    
    # Loss curve
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Val Loss', color='orange')
    plt.title(f"{model_name} Loss (Split: {split*100}%)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # Accuracy curve
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc', color='blue')
    plt.plot(history['val_acc'], label='Val Acc', color='orange')
    plt.title(f"{model_name} Accuracy (Split: {split*100}%)")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(f"Figure_Training_Curves_{model_name}_{int(split*100)}pct.png", dpi=300)
    plt.close('all')

def run_few_shot_experiments(X, y, bands=200, classes=16, epochs=50, num_seeds=10):
    from scipy.stats import ttest_rel, wilcoxon
    from sklearn.metrics import classification_report
    splits = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
    models = ['LightGBM', 'ViT', 'HybridSN']
    
    # Storage for Plotting (Mean Accuracies across seeds)
    curve_data_mean = {m: [] for m in models}
    curve_data_std = {m: [] for m in models}
    
    table_results = []
    
    print("=========================================================")
    print(f"🚀 STARTING {num_seeds}-SEED FEW-SHOT CURVES ON INDIAN PINES")
    print("=========================================================")

    for split in splits:
        print(f"\n{'='*40}\n--- Running Training Split: {split*100}% ---\n{'='*40}")
        
        split_accs = {m: [] for m in models}
        split_f1s = {m: [] for m in models}
        
        for seed in range(num_seeds):
            seed_everything(seed)
            print(f"  [Seed {seed+1}/{num_seeds}]")
            
            train_idx, test_idx = train_test_split(np.arange(len(y)), train_size=split, random_state=seed, stratify=y)
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            # --- LightGBM ---
            X_train_flat = X_train[:, 15//2, 15//2, :]
            X_test_flat = X_test[:, 15//2, 15//2, :]
            
            lgbm = lgb.LGBMClassifier(random_state=seed, n_estimators=100, n_jobs=-1, verbose=-1)
            start = time.time()
            lgbm.fit(X_train_flat, y_train)
            lgbm_train_time = time.time() - start
            
            start = time.time()
            y_pred_lgbm = lgbm.predict(X_test_flat)
            lgbm_inf_time = time.time() - start
            
            acc_lgbm = accuracy_score(y_test, y_pred_lgbm) * 100
            f1_lgbm = f1_score(y_test, y_pred_lgbm, average='macro')
            aa_lgbm = calc_average_accuracy(y_test, y_pred_lgbm)
            
            split_accs['LightGBM'].append(acc_lgbm)
            split_f1s['LightGBM'].append(f1_lgbm)
            
            if seed == 0:
                print(f"\n[LightGBM {int(split*100)}% Classification Report]")
                print(classification_report(y_test, y_pred_lgbm))
            
            # Prepare PyTorch Loaders
            train_loader = DataLoader(SoilDataset(X_train, y_train), batch_size=32, shuffle=True)
            test_loader = DataLoader(SoilDataset(X_test, y_test), batch_size=128, shuffle=False)
            
            # --- ViT ---
            vit = ViT1D(bands=bands, classes=classes).to(device)
            start = time.time()
            acc_vit, _, preds_vit, _, vit_history, vit_inf_time = train_pytorch_model(vit, train_loader, test_loader, epochs=epochs, patience=15)
            acc_vit = acc_vit * 100
            vit_train_time = time.time() - start
            
            f1_vit = f1_score(y_test, preds_vit, average='macro')
            aa_vit = calc_average_accuracy(y_test, preds_vit)
            
            split_accs['ViT'].append(acc_vit)
            split_f1s['ViT'].append(f1_vit)
            
            if seed == 0:
                print(f"\n[ViT {int(split*100)}% Classification Report]")
                print(classification_report(y_test, preds_vit))
                plot_training_curves(vit_history, "ViT", split)
            
            del vit
            gc.collect()
            torch.cuda.empty_cache()
            
            # --- HybridSN ---
            hsn = HybridSN(bands=bands, classes=classes, spatial_size=15).to(device)
            start = time.time()
            acc_hsn, _, preds_hsn, _, hsn_history, hsn_inf_time = train_pytorch_model(hsn, train_loader, test_loader, epochs=epochs, patience=15)
            acc_hsn = acc_hsn * 100
            hsn_train_time = time.time() - start
            
            f1_hsn = f1_score(y_test, preds_hsn, average='macro')
            aa_hsn = calc_average_accuracy(y_test, preds_hsn)
            
            split_accs['HybridSN'].append(acc_hsn)
            split_f1s['HybridSN'].append(f1_hsn)
            
            if seed == 0:
                print(f"\n[HybridSN {int(split*100)}% Classification Report]")
                print(classification_report(y_test, preds_hsn))
                plot_training_curves(hsn_history, "HybridSN", split)
            
            del hsn
            gc.collect()
            torch.cuda.empty_cache()
            
            # We only append to table_results on the final seed to avoid massive tables, or we average them later
            if seed == num_seeds - 1:
                table_results.append({'Split': split, 'Model': 'LightGBM', 'OA': np.mean(split_accs['LightGBM']), 'F1': np.mean(split_f1s['LightGBM']), 'Train Time': lgbm_train_time, 'Inf Time': lgbm_inf_time})
                table_results.append({'Split': split, 'Model': 'ViT', 'OA': np.mean(split_accs['ViT']), 'F1': np.mean(split_f1s['ViT']), 'Train Time': vit_train_time, 'Inf Time': vit_inf_time})
                table_results.append({'Split': split, 'Model': 'HybridSN', 'OA': np.mean(split_accs['HybridSN']), 'F1': np.mean(split_f1s['HybridSN']), 'Train Time': hsn_train_time, 'Inf Time': hsn_inf_time})

        # Calculate Means and Stds for the curve
        for m in models:
            mean_acc = np.mean(split_accs[m])
            std_acc = np.std(split_accs[m])
            curve_data_mean[m].append(mean_acc)
            curve_data_std[m].append(std_acc)
            print(f"  Final {m} -> OA: {mean_acc:.2f}% ± {std_acc:.2f}%")
            
        # Statistical Testing (Paired t-test and Wilcoxon)
        print("\n  [Statistical Significance Tests]")
        t_stat, p_val_t = ttest_rel(split_accs['LightGBM'], split_accs['HybridSN'])
        w_stat, p_val_w = wilcoxon(split_accs['LightGBM'], split_accs['HybridSN'])
        print(f"  LightGBM vs HybridSN -> T-Test p-value: {p_val_t:.4f} | Wilcoxon p-value: {p_val_w:.4f}")

    # Plotting the Curve with Error Bars
    plt.figure(figsize=(10, 6))
    splits_pct = [s * 100 for s in splits]
    
    plt.errorbar(splits_pct, curve_data_mean['LightGBM'], yerr=curve_data_std['LightGBM'], fmt='-o', linewidth=2, label='LightGBM', color='#ff7f0e', capsize=4)
    plt.errorbar(splits_pct, curve_data_mean['HybridSN'], yerr=curve_data_std['HybridSN'], fmt='-s', linewidth=2, label='HybridSN (3D-CNN)', color='#2ca02c', capsize=4)
    plt.errorbar(splits_pct, curve_data_mean['ViT'], yerr=curve_data_std['ViT'], fmt='-^', linewidth=2, label='ViT (Transformer)', color='#1f77b4', capsize=4)
    
    plt.title(f"Few-Shot Robustness ({num_seeds}-Seed Avg): Data Split vs Overall Accuracy", fontsize=14, fontweight='bold')
    plt.xlabel("Training Data Percentage (%)", fontsize=12)
    plt.ylabel("Overall Accuracy (%)", fontsize=12)
    plt.xticks(splits_pct)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=11)
    
    plt.savefig("Figure_FewShot_Curve_Robust.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close('all')
    print("✅ Graph saved as Figure_FewShot_Curve_Robust.png")



# ==========================================
# EXP 2 - CELL 3: 10-Seed Rigorous Tinto Evaluation
# ==========================================
# Note: Ensure your `X` and `y` are loaded from the Tinto dataset before running this.
# This directly addresses the reviewer's concern: "Tinto Benchmark Is Weakly Evaluated (only 1 seed)"

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score

def run_tinto_10_seed(X, y, bands, classes, num_seeds=10, epochs=50):
    models = ['LightGBM', '1D-CNN', 'ViT']
    
    # Storage for results
    results = {m: {'acc': [], 'kappa': []} for m in models}

    print("=========================================================")
    print(f"🚀 STARTING {num_seeds}-SEED EVALUATION ON TINTO DATASET")
    print("=========================================================")

    for seed in range(42, 42 + num_seeds):
        print(f"\n--- Running Seed: {seed} ---")
        
        # Standard 30% testing split for Tinto (or adjust as needed)
        train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.30, random_state=seed, stratify=y)
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # --- LightGBM ---
        # Assuming X is shape (B, H, W, C)
        X_train_flat = X_train[:, X_train.shape[1]//2, X_train.shape[2]//2, :]
        X_test_flat = X_test[:, X_test.shape[1]//2, X_test.shape[2]//2, :]
        
        lgbm = lgb.LGBMClassifier(random_state=seed, n_estimators=100, n_jobs=-1)
        lgbm.fit(X_train_flat, y_train)
        y_pred_lgbm = lgbm.predict(X_test_flat)
        
        acc_lgbm = accuracy_score(y_test, y_pred_lgbm) * 100
        kap_lgbm = cohen_kappa_score(y_test, y_pred_lgbm)
        results['LightGBM']['acc'].append(acc_lgbm)
        results['LightGBM']['kappa'].append(kap_lgbm)
        
        # --- PyTorch Setup ---
        train_loader = DataLoader(SoilDataset(X_train, y_train), batch_size=32, shuffle=True)
        test_loader = DataLoader(SoilDataset(X_test, y_test), batch_size=128, shuffle=False)
        
        # --- 1D-CNN ---
        cnn = CNN1D(bands=bands, classes=classes).to(device)
        acc_cnn, kap_cnn, _, _, _, _ = train_pytorch_model(cnn, train_loader, test_loader, epochs=epochs, patience=15)
        acc_cnn = acc_cnn * 100
        results['1D-CNN']['acc'].append(acc_cnn)
        results['1D-CNN']['kappa'].append(kap_cnn)
        del cnn
        gc.collect()
        torch.cuda.empty_cache()

        # --- ViT ---
        vit = ViT1D(bands=bands, classes=classes).to(device)
        acc_vit, kap_vit, _, _, _, _ = train_pytorch_model(vit, train_loader, test_loader, epochs=epochs, patience=15)
        acc_vit = acc_vit * 100
        results['ViT']['acc'].append(acc_vit)
        results['ViT']['kappa'].append(kap_vit)
        del vit
        gc.collect()
        torch.cuda.empty_cache()

    # Aggregate Results
    print("\n" + "="*50)
    print(f"FINAL {num_seeds}-SEED RESULTS (Tinto Dataset)")
    print("="*50)
    final_df = []
    for m in models:
        mean_acc = np.mean(results[m]['acc'])
        std_acc = np.std(results[m]['acc'])
        mean_kap = np.mean(results[m]['kappa'])
        std_kap = np.std(results[m]['kappa'])
        final_df.append({'Model': m, 'OA (%)': f"{mean_acc:.2f} ± {std_acc:.2f}", 'Kappa': f"{mean_kap:.4f} ± {std_kap:.4f}"})
        
    df = pd.DataFrame(final_df)
    print(df.to_string(index=False))
    df.to_csv("Tinto_10_Seed_Results.csv", index=False)
    print("\n✅ Results saved to Tinto_10_Seed_Results.csv")

# ==========================================
# USAGE INSTRUCTIONS FOR KAGGLE
# ==========================================
# Ensure you have run data extraction using the following dataset path:
# TINTO_DIR = "/kaggle/input/datasets/dev123123456/tinto-hyperspectral-digital-outcrop"
#
# Once X_tinto and y_tinto are loaded, run:
# run_tinto_10_seed(X_tinto, y_tinto, bands=tinto_bands, classes=tinto_classes)


# ==========================================
# EXP 2 - CELL 4: Transfer Learning Experiment (Local Soil -> Indian Pines)
# ==========================================
# This directly addresses the reviewer's concern: "Why not pretrain ViT? Train: Local Soil -> Indian Pines"

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def run_transfer_learning(X_local, y_local, X_indian, y_indian, bands_local=168, classes_local=6, bands_indian=200, classes_indian=16):
    print("=========================================================")
    print("🚀 STARTING TRANSFER LEARNING EXPERIMENT")
    print("=========================================================")
    
    # 1. Train ViT on Local Soil Dataset (Pre-training)
    print("\n--- Phase 1: Pre-training ViT on Local Soil (Abundant Data) ---")
    train_loader_local = DataLoader(SoilDataset(X_local, y_local), batch_size=64, shuffle=True)
    # We will use all of Local Soil to pre-train, so we'll just test on a small subset or skip formal testing
    
    vit_pretrained = ViT1D(bands=bands_local, classes=classes_local).to(device)
    
    # Train function modified slightly to just run epochs without early stopping if no test set
    optimizer = torch.optim.Adam(vit_pretrained.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    vit_pretrained.train()
    for epoch in range(10): # 10 epochs of pretraining is sufficient to learn spectral weights
        for batch_x, batch_y in train_loader_local:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            out = vit_pretrained(batch_x)
            loss = criterion(out, batch_y)
            loss.backward()
            optimizer.step()
    print("✅ Pre-training Complete.")
    
    # 2. Modify ViT for Indian Pines (Fine-tuning)
    # Note: If the number of bands differs (168 vs 200), we cannot easily transfer the very first layer.
    # However, since they represent different spectral resolutions, true transfer learning requires interpolating the input layer.
    # For a simplified but valid approach: We freeze/transfer the core Transformer blocks!
    
    print("\n--- Phase 2: Fine-Tuning on Indian Pines (5% Few-Shot Split) ---")
    
    # Stratified 5% split for Indian Pines
    train_idx, test_idx = train_test_split(np.arange(len(y_indian)), train_size=0.05, random_state=42, stratify=y_indian)
    X_train_ip, X_test_ip = X_indian[train_idx], X_indian[test_idx]
    y_train_ip, y_test_ip = y_indian[train_idx], y_indian[test_idx]
    
    train_loader_ip = DataLoader(SoilDataset(X_train_ip, y_train_ip), batch_size=32, shuffle=True)
    test_loader_ip = DataLoader(SoilDataset(X_test_ip, y_test_ip), batch_size=128, shuffle=False)
    
    # Create a new ViT for Indian Pines
    vit_finetune = ViT1D(bands=bands_indian, classes=classes_indian).to(device)
    
    # Transfer the Transformer Encoder weights!
    vit_finetune.transformer.load_state_dict(vit_pretrained.transformer.state_dict())
    
    # (Optional) Freeze the transformer blocks to prevent overfitting on 5% data
    # for param in vit_finetune.transformer.parameters():
    #     param.requires_grad = False
        
    print("[Training Pre-trained ViT on Indian Pines...]")
    acc_finetune, kap_finetune, _, _, _, _ = train_pytorch_model(vit_finetune, train_loader_ip, test_loader_ip, epochs=50, patience=15)
    acc_finetune = acc_finetune * 100
    
    # Compare with scratch ViT
    print("\n[Training Scratch ViT on Indian Pines...]")
    vit_scratch = ViT1D(bands=bands_indian, classes=classes_indian).to(device)
    acc_scratch, kap_scratch, _, _, _, _ = train_pytorch_model(vit_scratch, train_loader_ip, test_loader_ip, epochs=50, patience=15)
    acc_scratch = acc_scratch * 100
    
    print("\n" + "="*50)
    print("FINAL TRANSFER LEARNING RESULTS")
    print("="*50)
    print(f"ViT (Trained from Scratch) -> OA: {acc_scratch:.2f}%")
    print(f"ViT (Pre-trained on Local) -> OA: {acc_finetune:.2f}%")
    
    if acc_finetune > acc_scratch:
        print("\nInsight: Pre-training effectively transferred spectral extraction capabilities, overcoming the few-shot limitation!")
    else:
        print("\nInsight: Pre-training did not overcome the few-shot gap. LightGBM remains the superior choice.")
        
    del vit_pretrained
    del vit_finetune
    del vit_scratch
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# USAGE INSTRUCTIONS FOR KAGGLE
# ==========================================
# Ensure you have run data extraction using the following dataset paths:
# LOCAL_SOIL_DIR = "/kaggle/input/datasets/dev123123456/local-soil-hyperspectral-dataset"
#
# Once X_local, y_local, X_indian, and y_indian are loaded, run:
# run_transfer_learning(X_local, y_local, X_indian, y_indian)



# ==========================================
# FINAL EXECUTION BLOCK (AUTOMATICALLY RUNS EVERYTHING)
# ==========================================
if __name__ == '__main__':
    LOCAL_SOIL_DIR = '/kaggle/input/datasets/dev123123456/local-soil-hyperspectral-dataset'
    TINTO_DIR = '/kaggle/input/datasets/dev123123456/tinto-hyperspectral-digital-outcrop'
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using device:', device)

    # 1. Load Data
    X_local, y_local, _ = load_local_dataset(LOCAL_SOIL_DIR)
    
    public_data = load_public_datasets()
    
    # We need spatial patches (15x15) for HybridSN, not just 1D pixels!
    print("⏳ Extracting 15x15 spatial patches from Indian Pines for HybridSN...")
    ip_img = public_data['IndianPines']['img']
    ip_gt = scipy.io.loadmat('Indian_pines_gt.mat')['indian_pines_gt']
    
    def extract_ip_patches(img_cube, gt, patch_size=15):
        h, w, c = img_cube.shape
        margin = patch_size // 2
        img_padded = np.pad(img_cube, ((margin, margin), (margin, margin), (0, 0)), mode='reflect')
        patches, labels = [], []
        for i in range(h):
            for j in range(w):
                if gt[i, j] > 0:
                    patches.append(img_padded[i:i+patch_size, j:j+patch_size, :])
                    labels.append(gt[i, j] - 1)
        return np.array(patches, dtype=np.float32), np.array(labels)
        
    X_indian, y_indian = extract_ip_patches(ip_img, ip_gt, patch_size=15)
    print(f"✅ Indian Pines Patches Extracted: Shape {X_indian.shape}")
    
    indian_bands = public_data['IndianPines']['bands']
    indian_classes = public_data['IndianPines']['classes']
    
    # 2. FLOPs Benchmark
    vit_input = (1, 15, 15, indian_bands)
    vit_model = ViT1D(bands=indian_bands, classes=indian_classes).to(device)
    hsn_input = (1, 15, 15, indian_bands)
    hsn_model = HybridSN(bands=indian_bands, classes=indian_classes, spatial_size=15).to(device)
    print('\n--- FLOPs Benchmark ---')
    calculate_flops_and_params(vit_model, vit_input)
    calculate_flops_and_params(hsn_model, hsn_input)
    
    # 3. Few-Shot Curves
    run_few_shot_experiments(X_indian, y_indian, bands=indian_bands, classes=indian_classes, epochs=50)
    
    # 4. Transfer Learning
    run_transfer_learning(X_local, y_local, X_indian, y_indian, bands_local=168, classes_local=3, bands_indian=indian_bands, classes_indian=indian_classes)
    
    # Note: Tinto loading logic from local path is custom, assuming a specific load structure.
    # If Tinto requires a specific loading function not present in load_public_datasets,
    # the user will need to adjust. Assuming it's in the original Kaggle notebook, we skip automated Tinto execution
    # to avoid pathing crashes, but the function is available.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.2 MB ? eta -:--:--

   ━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/12.2 MB 39.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━ 7.7/12.2 MB 73.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 12.1/12.2 MB 148.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 82.0 MB/s eta 0:00:00


...

  [Seed 5/10]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [Seed 6/10]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


KeyboardInterrupt: 